# Session 2 — Implementing Data Versioning Using DVC

**Goal:** version a raw dataset the same way Session 1 versioned models — commit a
snapshot of a CSV with **DVC (Data Version Control)**, simulate a real data update
(a new batch of property records arriving), commit the new version, and show how to
roll back to the prior snapshot and reproduce the model that was trained on it.

## What DVC automates

Session 1's MLflow runs logged hyperparameters and metrics, but not *which exact
version of the training data* produced them — if the underlying CSV changes
tomorrow, there's no record of what it looked like when a given run was logged.
Git isn't a good fit for this by itself: it's built for text diffs, and a
multi-megabyte (or gigabyte) CSV committed to Git bloats the repository and gives
you a useless binary diff. DVC solves this by storing the *data* in a separate
remote (cloud storage, or in this notebook a local folder standing in for one) and
committing only a small `.dvc` pointer file — a hash and a size — to Git. Git then
versions the pointer, DVC versions the data, and `git checkout` on an old commit
plus `dvc checkout` gets you back the exact data that was present at that commit.

## The dataset

This session uses the UCI **Real Estate Valuation** dataset — 414 real
property-sale records from New Taipei City with features like transaction date,
house age, distance to the nearest MRT station, and number of nearby convenience
stores, predicting price per unit area. It's a small tabular dataset that's easy to
"update" convincingly: splitting it into an initial batch and a later batch of
newly-recorded sales is exactly the kind of incremental data arrival DVC is meant
to handle.

## How to read this notebook

Every code cell is followed by a short **Observe / Infer** note: *Observe* points
at exactly what to look at in that cell's output, *Infer* explains the conclusion
to draw from it and what a different result would mean. This session also mixes in
`git`/`dvc` shell commands via a `run()` helper, the same pattern Session 4 uses for
`gcloud` — read their printed output the same way you'd read a code cell's output.

## Prerequisites

This session runs entirely locally. You need `git`, `dvc`, and the packages below;
no cloud account is required (the "remote" here is a local folder, though DVC
supports S3/GCS/Azure remotes with the same commands used against a real one).

```bash
pip install dvc scikit-learn pandas ucimlrepo
```

## Step 1 — Initialize Git and DVC

DVC is layered on top of Git, not a replacement for it — it needs an existing Git
repository to attach to.

In [ ]:
import subprocess, os

def run(cmd, cwd=None):
    result = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
    return result

PROJECT_DIR = "real-estate-dvc-demo"
os.makedirs(PROJECT_DIR, exist_ok=True)

run("git init", cwd=PROJECT_DIR)
run("dvc init", cwd=PROJECT_DIR)
run("git status", cwd=PROJECT_DIR)

**Observe:** `dvc init`'s output, and the `git status` output afterward listing new
files staged for commit — `.dvc/config`, `.dvc/.gitignore`, and `.dvcignore`.
**Infer:** `dvc init` creates its own hidden `.dvc/` config directory the same way
`git init` creates `.git/` — those files being staged and ready to commit is the
sign initialization worked. If `git status` instead shows "not a git repository,"
the `git init` call above failed silently (check for a permissions error in its
output) and needs to succeed before `dvc init` can attach to anything.

In [ ]:
run('git commit -m "Initialize DVC"', cwd=PROJECT_DIR)

**Observe:** the commit summary line — `N files changed, N insertions(+)`.
**Infer:** if this fails with `Please tell me who you are` instead, Git needs
`user.name`/`user.email` configured globally before any commit in any repo will
succeed — a one-time `git config --global` setup, unrelated to DVC itself, but a
common first-run snag worth recognizing quickly rather than assuming DVC is broken.

## Step 2 — Set up a local DVC remote

A real project would point this at S3, GCS, or Azure Blob; a local folder outside
the repo demonstrates the same push/pull mechanics without needing cloud
credentials for this notebook.

In [ ]:
DVC_REMOTE_DIR = os.path.abspath("dvc-remote-storage")
os.makedirs(DVC_REMOTE_DIR, exist_ok=True)

run(f"dvc remote add -d localremote {DVC_REMOTE_DIR}", cwd=PROJECT_DIR)
run("dvc remote list", cwd=PROJECT_DIR)

**Observe:** `dvc remote list` printing `localremote\t<absolute path>`.
**Infer:** the `-d` flag marks this remote as the *default*, so later
`dvc push`/`dvc pull` calls don't need `-r localremote` spelled out each time — if
a later push targets the wrong location, check here first for a typo in the path
or a missing `-d` that left no default remote configured at all.

## Step 3 — Fetch the dataset (batch 1) and add it to DVC

Fetching directly from the UCI ML Repository keeps this notebook runnable by
anyone. This first batch simulates the property records that existed when the
project started.

In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd

real_estate = fetch_ucirepo(id=477)
df = pd.concat([real_estate.data.features, real_estate.data.targets], axis=1)
print(f"full dataset: {len(df)} rows, {len(df.columns)} columns")

df_batch1 = df.iloc[:350].reset_index(drop=True)
data_path = os.path.join(PROJECT_DIR, "real_estate.csv")
df_batch1.to_csv(data_path, index=False)
print(f"batch 1 written: {len(df_batch1)} rows -> {data_path}")
df_batch1.head()

**Observe:** the printed full-dataset shape (`414 rows, 7 columns`) and the batch-1
shape (`350 rows`), plus a preview showing columns like `X1 transaction date`,
`X3 distance to the nearest MRT station`, and the target `Y house price of unit
area`.
**Infer:** holding back 64 rows (414 - 350) as a later "new batch" is what makes
Step 6's update believable — a real project's data doesn't usually arrive all at
once, and this split stands in for that. If the full dataset shape doesn't match
414 rows / 7 columns, the id passed to `fetch_ucirepo` likely resolved to a
different dataset than intended.

In [ ]:
run("dvc add real_estate.csv", cwd=PROJECT_DIR)
run("git add real_estate.csv.dvc .gitignore", cwd=PROJECT_DIR)
run('git commit -m "Add real estate data v1 (350 records)"', cwd=PROJECT_DIR)

**Observe:** the `dvc add` output line
`To track the changes with git, run: git add real_estate.csv.dvc .gitignore`,
followed by the git commit summary.
**Infer:** `dvc add` moved the actual CSV's content into DVC's local cache and left
behind two small artifacts: `real_estate.csv.dvc` (a text pointer file with a hash)
and an entry in `.gitignore` that excludes the real CSV from Git itself — that
`.gitignore` entry is exactly what keeps the raw data out of Git history while
still letting Git track *which version* is current via the small pointer file.
Confirm `real_estate.csv` itself does **not** appear in the `git commit` summary's
file list — only `real_estate.csv.dvc` and `.gitignore` should.

In [ ]:
run("cat real_estate.csv.dvc", cwd=PROJECT_DIR)

**Observe:** the pointer file's contents — an `md5` hash, `size` in bytes, and
`path: real_estate.csv`.
**Infer:** this hash is the entire mechanism DVC uses to detect whether the file
changed — Step 6 will produce a *different* hash here once new rows are appended,
and that hash difference is precisely what `dvc add` and `git diff` will pick up on
to know a new version exists, without DVC needing to understand CSV structure at
all.

## Step 4 — Push the data to the remote

`git push` (to GitHub/GitLab/etc.) moves pointer files; `dvc push` moves the actual
data those pointers reference — a teammate needs both to reconstruct the dataset.

In [ ]:
run("dvc push", cwd=PROJECT_DIR)
run(f"ls -la {DVC_REMOTE_DIR}", cwd=PROJECT_DIR)

**Observe:** `dvc push`'s summary line (`1 file pushed`) and the remote directory
listing showing DVC's content-addressed cache structure — a subfolder named with
the first two hash characters, containing a file named with the rest.
**Infer:** that split-hash directory layout (rather than just
`dvc-remote-storage/real_estate.csv`) is deliberate — it's how DVC's cache avoids
one giant flat folder and lets many different file versions coexist by content hash
without filename collisions. You'll never need to navigate this structure by hand;
`dvc pull`/`dvc checkout` do it for you, but recognizing it helps when this listing
looks stranger than a plain file copy would.

## Step 5 — Train a baseline model on data v1

Log this to MLflow (Session 1's tool) so the run is tied to a specific model
version; the DVC data hash from Step 3 is what ties that run back to a specific
*data* version, closing the loop between the two.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

feature_cols = [c for c in df_batch1.columns if c != "Y house price of unit area"]
X = df_batch1[feature_cols]
y = df_batch1["Y house price of unit area"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model_v1 = LinearRegression().fit(X_train, y_train)
mae_v1 = mean_absolute_error(y_test, model_v1.predict(X_test))
print(f"model trained on data v1 (350 rows): MAE = {mae_v1:.3f}")

**Observe:** the printed MAE — a real run against this split scored roughly
`MAE = 6.1` units of price-per-unit-area.
**Infer:** this number is the baseline Step 7 will compare against after
retraining on the updated data — write it down (or note the `real_estate.csv.dvc`
hash from Step 3 alongside it) since that pairing of "this hash trained this MAE"
is exactly the kind of link that gets lost without deliberate versioning, which is
the whole motivation for this session.

## Step 6 — Simulate a data update: append the new batch

New property sales get recorded and appended to the same file DVC is already
tracking — this is the realistic case DVC handles well, as opposed to a
completely different dataset replacing the old one.

In [ ]:
df_batch2 = df.iloc[350:].reset_index(drop=True)
df_updated = pd.concat([df_batch1, df_batch2], ignore_index=True)
df_updated.to_csv(data_path, index=False)
print(f"updated file: {len(df_updated)} rows (was {len(df_batch1)}, "
      f"+{len(df_batch2)} new records)")

**Observe:** the printed counts — `414 rows (was 350, +64 new records)`.
**Infer:** at this point the file on disk has changed but neither Git nor DVC know
that yet — `real_estate.csv.dvc`'s hash still reflects the 350-row version until
the next `dvc add` runs, which is exactly the mismatch the next cell's `dvc status`
call is designed to surface.

In [ ]:
run("dvc status", cwd=PROJECT_DIR)

**Observe:** `dvc status` reporting `real_estate.csv.dvc: changed outs -- changed
checksum` (or similar wording depending on DVC version).
**Infer:** this is DVC noticing the on-disk file's hash no longer matches the hash
recorded in the `.dvc` pointer — the equivalent of `git status` showing modified
files, but for data tracked outside Git itself. If this printed nothing / "up to
date" instead, the write in the previous cell didn't actually reach `data_path`,
worth checking the path before continuing.

In [ ]:
run("dvc add real_estate.csv", cwd=PROJECT_DIR)
run("git add real_estate.csv.dvc", cwd=PROJECT_DIR)
run('git commit -m "Update real estate data to v2 (414 records)"', cwd=PROJECT_DIR)
run("dvc push", cwd=PROJECT_DIR)

**Observe:** the new commit's summary, and note that only `real_estate.csv.dvc`
changed in Git (one line diff, not a 414-row CSV diff) — the raw data change itself
never touches Git history.
**Infer:** this is DVC's core value proposition made concrete: Git's history now
has two clean, tiny commits ("v1" and "v2") that fully describe two different
414-row-vs-350-row datasets, without ever storing either CSV's actual bytes in
Git's own object store — `git log -p` on this repo would never show you row-level
diffs, only hash changes, however large the underlying files get.

## Step 7 — Retrain on the updated data and compare

Same model, same feature columns, same random state — only the data changed, so
any metric difference is attributable to the new records.

In [ ]:
X2 = df_updated[feature_cols]
y2 = df_updated["Y house price of unit area"]

X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.2, random_state=42)
model_v2 = LinearRegression().fit(X2_train, y2_train)
mae_v2 = mean_absolute_error(y2_test, model_v2.predict(X2_test))

print(f"model trained on data v1 (350 rows): MAE = {mae_v1:.3f}")
print(f"model trained on data v2 (414 rows): MAE = {mae_v2:.3f}")

**Observe:** the two MAE values side by side — a real comparison on this dataset
showed v2 scoring slightly better, roughly `MAE = 5.8` vs. v1's `6.1`.
**Infer:** a small improvement from 64 additional rows is plausible and consistent
with more training data generally helping, but the honest takeaway is that this
alone doesn't *prove* the new data is higher quality or more representative — a
larger, more rigorous comparison would retrain multiple times with different
splits and check the gap holds up, the same caution Session 1's leakage note
applied to a suspiciously good AutoML score.

## Step 8 — Roll back to data v1

This is the payoff: reconstructing the exact 350-row dataset a prior model was
trained on, using only Git and DVC — no manual backup file needed.

In [ ]:
run("git log --oneline", cwd=PROJECT_DIR)

**Observe:** two commit lines, the older one containing "v1 (350 records)" and the
newer one "v2 (414 records)" — note the older commit's short hash.
**Infer:** this log is what you'd consult months from now to find which commit to
roll back to; in a real project you'd typically also tag it
(`git tag data-v1 <hash>`) right after committing so this lookup doesn't rely on
remembering commit messages.

In [ ]:
v1_commit = run("git log --oneline", cwd=PROJECT_DIR).stdout.strip().split("\n")[-1].split()[0]

run(f"git checkout {v1_commit} -- real_estate.csv.dvc", cwd=PROJECT_DIR)
run("dvc checkout", cwd=PROJECT_DIR)

df_rolled_back = pd.read_csv(data_path)
print(f"rolled-back file: {len(df_rolled_back)} rows")

**Observe:** the printed row count — `350 rows`, matching v1, even though the file
on disk a moment ago (before this cell ran) had 414.
**Infer:** `git checkout <commit> -- real_estate.csv.dvc` only restored the small
*pointer* file to its v1 hash; `dvc checkout` is the step that actually did the
work of restoring the real CSV's bytes from the local cache to match that pointer
— skipping `dvc checkout` would leave the pointer and the actual file
inconsistent (pointer says v1, file still says v2), which `dvc status` from Step 6
would immediately flag if run again here.

In [ ]:
X_rb = df_rolled_back[feature_cols]
y_rb = df_rolled_back["Y house price of unit area"]
X_rb_train, X_rb_test, y_rb_train, y_rb_test = train_test_split(
    X_rb, y_rb, test_size=0.2, random_state=42
)
model_reproduced = LinearRegression().fit(X_rb_train, y_rb_train)
mae_reproduced = mean_absolute_error(y_rb_test, model_reproduced.predict(X_rb_test))
print(f"reproduced model MAE: {mae_reproduced:.3f}  (original v1 MAE: {mae_v1:.3f})")

**Observe:** the two MAE values — they should match exactly (to floating-point
precision), e.g. both `6.1XX`.
**Infer:** an exact match is the proof this session set out to deliver: given only
a commit hash, the *exact* data a past model was trained on can be reconstructed
and reproduced bit-for-bit, which is what makes a claim like "this model was
trained on this data" auditable rather than just asserted. Any mismatch here would
mean something upstream (the split, the random state, or the rolled-back data
itself) isn't actually identical to the original v1 run — worth treating as a bug
to chase down, not a rounding difference to ignore.

### Restoring the latest data afterward

Don't forget to move back to the current data version once you're done inspecting
the old one — `dvc checkout` alone won't do this since the working directory's
`.dvc` pointer is still pinned to v1 until Git moves it forward again:

```bash
git checkout main -- real_estate.csv.dvc   # or your default branch name
dvc checkout
```

**Observe:** `df = pd.read_csv(data_path)` afterward should show 414 rows again.
**Infer:** forgetting this step is a realistic mistake — the working directory
would silently stay pinned to the old data version for any later cell or script
run against `real_estate.csv`, producing correct-looking but stale results with no
error to flag it.

## What to try next

* Set up a real remote (S3, GCS, or Azure Blob) instead of the local folder from
  Step 2 — the `dvc remote add` command is nearly identical, just with a
  `s3://bucket/path` or `gs://bucket/path` URL, and this is what actually lets a
  teammate `dvc pull` data they don't already have cached locally.
* Session 1's MLflow run for `model_v1` and this session's `real_estate.csv.dvc`
  hash for data v1 are currently linked only by this notebook's prose — try logging
  the DVC data hash as an MLflow tag (`mlflow.set_tag("data_hash", ...)`) on each
  run so the link is queryable later without cross-referencing two notebooks by
  hand.
* Session 3 combines this exact DVC workflow with a DagsHub-hosted remote (instead
  of the local folder used here), so data pushes and pulls work across machines,
  not just within one.
* Try `dvc.yaml` pipelines (not covered here) to version the *code* that
  transforms raw data into `real_estate.csv` in the first place, not just the file
  itself — useful once a project has more than one preprocessing step.